# Structure recovery

**Difficulty:** Beginner

Reconstruct a document's structure from plain text that lost its formatting — hard-wrapped mid-sentence, no headings, no bullets — in two passes: one `Noul` per adjacent line pair to heal the wraps, one `Choice` per resulting block to classify it. The model never generates text; every character of the output comes from the input.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/autoformat (click to expand)</summary>

# Structure recovery

> Reconstructs Markdown from plain text that lost its formatting in two requests: one stitches hard-wrapped lines back together, one classifies every block (heading, list, code, callout) with companion questions read only when relevant.

This cookbook takes plain text whose markup has been stripped (lines hard-wrapped mid-sentence, no heading markers, no list bullets) and reconstructs the structure as Markdown. A text-generation model could rewrite the text into Markdown, but a rewrite can also change the words. Here the model never generates text: it answers narrow questions about the document, and code does the rendering — every character of the output comes from the input, and every judgment carries a probability.

Two requests, run in sequence:

- **Pass 1, stitch:** one `Noul` per adjacent pair of lines, asking whether the line break split a sentence across the two. All the pairs go in a single request; lines that continue a split sentence get merged back into blocks.
- **Pass 2, classify:** one `Choice` per merged block, choosing among `heading` / `paragraph` / `list_item` / `quote` / `code` / `callout`. This is a second request, since the blocks don't exist until pass 1 has answered — but it also carries companion questions (heading level, step order for list items, callout kind) speculatively, in the same request, read only when the block's type makes them relevant. Blank lines and explicit markers (`- `, `1.`, `#`) are read in code, never sent to the model — the model only gets questions code genuinely can't answer from the text.

## Pass 1: stitching, worked example

```
join_question(i) = Noul(
    instructions=f"Does line {line_id(i)} pick up mid-sentence, continuing a sentence left "
                 f"unfinished at the end of line {line_id(i-1)}?",
    criteria={
        "true": "The line starts in the middle of a sentence that began on the previous line",
        "false": "The line begins a new sentence, item, heading, or thought of its own",
    },
)
```

On the cookbook's test memo: 16 pair questions, one request, 0.32s. The merge cutoff depends on how the previous line ends — after a dangling line (no sentence-ending punctuation), a join probability of `0.2` or above merges the pair; after terminal punctuation (`.` `!` `?` `:` `;`), the cutoff rises to `0.5`. Result: 28 lines → 17 blocks, 11 line breaks healed.

## Pass 2: classifying blocks, worked example

```
TYPE_CRITERIA = {
    "heading":   "A short label or title that names the document or section - not a full sentence",
    "paragraph": "Running prose: one or more complete sentences of explanatory text",
    "list_item": "One entry in a list of parallel items",
    "quote":     "Words attributed to a person or source",
    "code":      "Computer code, a shell command, or a config snippet meant to be read verbatim",
    "callout":   "A warning, tip, or important note that interrupts the flow",
}
```

62 questions about 17 blocks (`type` + speculative companions), one request, 0.51s:

```
block type       conf  companion used    text
B000  heading    0.99  level=title       Migration to the new build system
B001  paragraph  0.98  -                 Hi everyone, quick heads up about the build sy...
B004  code       1.00  -                 bun run build
B007  list_item  0.99  step=0.15         The platform team
B011  list_item  0.98  step=0.86         Update your local toolchain to version 2.4 or ...
B014  callout    0.65  kind=warning      If the doctor script reports a red result on t...
```

The three "team" list items carry `step` probabilities near `0.15` (order doesn't matter → bulleted list); the three "things to do" list items carry `step` near `0.9` (order matters → numbered list); an unmarked warning about a doctor script was classified as a `callout` of kind `warning` purely from its content.

> The full cookbook fetches a real team memo from a pinned gist, adds a full rendering step back to Markdown, and includes an appendix on where the join thresholds (`0.2` / `0.5`) come from — trimmed here to the two-pass mechanic itself. Read the full cookbook at [docs.typesafe.ai/cookbooks/autoformat](https://docs.typesafe.ai/cookbooks/autoformat) any time.

</details>

## The core idea

Two narrow questions, asked twice, doing a job that looks like it needs a rewrite: `Noul` decides *where the blocks are* (a fact about line breaks, closer to objective than most judgments you've asked so far), `Choice` decides *what each block is*. Two requests, not one, because the blocks pass 2 needs don't exist until pass 1 has answered — a real case where a second round trip is warranted, unlike the "just ask everything up front" default from speculative fan-out. What doesn't change: within each pass, every question still goes in one request, and pass 2's companion questions are asked speculatively for every block regardless of type, exactly like lesson 6.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { noul, choice } from "npm:@typesafe-ai/sdk";

// Plain text that lost its formatting: hard-wrapped mid-sentence, no headers, no bullets.
const rawLines = [
  "Migration to the new build system",
  "Hi everyone, quick heads up about the build system migration that is",
  "happening next week. Please update your local toolchain before then.",
  "Update your local toolchain to version 2.4 or later",
  "Delete the old build cache directory",
];

function lineId(i: number): string {
  return `L${String(i).padStart(3, "0")}`;
}

const taggedLines = rawLines.map((line, i) => `${lineId(i)}| ${line}`).join("\n");

// Pass 1: one Noul per adjacent pair, all in one request.
const stitchQuestions = Object.fromEntries(
  rawLines.slice(1).map((_, idx) => {
    const i = idx + 1;
    return [
      lineId(i),
      noul(
        `Does line ${lineId(i)} pick up mid-sentence, continuing a sentence left unfinished at the end of line ${lineId(i - 1)}?`,
        {
          true: "The line starts in the middle of a sentence that began on the previous line",
          false: "The line begins a new sentence, item, heading, or thought of its own",
        },
      ),
    ];
  }),
);

const stitchResponse = await client.systemOne({ state: taggedLines, questions: stitchQuestions });

const JOIN_THRESHOLD = 0.5;
const blocks: string[] = [];
for (let i = 0; i < rawLines.length; i++) {
  const join = i > 0 ? stitchResponse.answers[lineId(i)].noul : 0;
  if (i > 0 && join >= JOIN_THRESHOLD) {
    blocks[blocks.length - 1] += " " + rawLines[i];
  } else {
    blocks.push(rawLines[i]);
  }
}

console.log(`${rawLines.length} lines -> ${blocks.length} blocks`);
blocks.forEach((b, i) => console.log(`B${i}: ${b}`));

// Pass 2: one Choice per merged block, second request (blocks didn't exist until pass 1 ran).
const TYPE_CRITERIA = {
  heading: "A short label or title that names the document or a section - not a full sentence of content",
  paragraph: "Running prose: one or more complete sentences of explanatory text",
  list_item: "One entry in a list of parallel items",
};

const classifyQuestions = Object.fromEntries(
  blocks.map((_, i) => [`type${i}`, choice("What kind of content is this block?", TYPE_CRITERIA)]),
);

const taggedBlocks = blocks.map((b, i) => `B${i}| ${b}`).join("\n");
const classifyResponse = await client.systemOne({ state: taggedBlocks, questions: classifyQuestions });

blocks.forEach((b, i) => {
  const answer = classifyResponse.answers[`type${i}`];
  console.log(`B${i} (${answer.choice}, conf ${answer.confidence.toFixed(2)}): ${b}`);
});

## What just happened

Five raw lines, two of which are really one sentence split by a hard wrap. Pass 1 should merge lines 1–2 into a single block (the wrapped sentence) and leave the title and the two list items alone — check `blocks.length`, it should come out to 4, not 5. Pass 2 then classifies each of those 4 blocks: expect `heading` for the title, `paragraph` for the merged sentence, `list_item` for the last two.

**Try it yourself:** add a sixth line that's a bare shell command (e.g. `"npm install"`) right after the list items, predict whether pass 1 merges it into the previous block or keeps it separate, and whether pass 2 classifies it as `code` even without a `TYPE_CRITERIA` entry that mentions commands explicitly by name — the description says "computer code... meant to be read verbatim," which should still match.

## Related

- `lessons/core-concepts/0002-choice.ipynb`, `lessons/core-concepts/0004-noul-and-picking-the-right-primitive.ipynb` — the two primitives this pipeline combines
- `lessons/patterns/0006-speculative-fan-out.ipynb` — why pass 2's companion questions are all asked up front instead of after seeing the type
- `cookbooks/how-to/line-by-line-search.ipynb` — another cookbook that tags lines with ids for the model to reference

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.